# 02 — Connect your Hugging Face token safely (BYOT)

Connect your own token. The token is NEVER printed — only `hf_***xx` is shown.

> VisionServeX does **not** redistribute gated or restricted model weights. You bring your own token and accept upstream licenses yourself. Tokens are always redacted.

In [ ]:
# Install the published package from PyPI (run AFTER release).
# Before release you may instead `pip install dist/visionservex-3.8.0-py3-none-any.whl`.
# %pip install -q visionservex==3.8.0
import importlib.metadata as _m
print("installed:", _m.version("visionservex"))

In [ ]:
# Assert we are using the INSTALLED package (site-packages), never the local src.
import visionservex
print("visionservex:", visionservex.__version__)
print("file:", visionservex.__file__)
assert "site-packages" in visionservex.__file__, (
    "This tutorial must run against the pip-installed package, not local src. "
    "Use a fresh venv / clean kernel and install visionservex from PyPI."
)

In [ ]:
from visionservex import VSX
status = VSX.hf.status()           # redacted token only
print({k: status.get(k) for k in ("logged_in", "token_source", "token_redacted", "name")})

CLI equivalents (token never printed):
```bash
huggingface-cli login            # or:
visionservex hf connect --token-env HF_TOKEN
visionservex hf status
visionservex hf whoami
```

In [ ]:
# Record an execution-ledger row (artifact or honest blocker).
import csv, json, os, time
from pathlib import Path
ART = Path("v38_tutorial_artifacts"); ART.mkdir(exist_ok=True)
def record(notebook, status, detail, artifact=None):
    art = None
    if artifact is not None:
        art = ART / f"{notebook}.json"
        art.write_text(json.dumps(artifact, indent=2, default=str))
    row = {"notebook": notebook, "status": status, "detail": detail,
           "artifact": str(art) if art else "", "version": __import__("visionservex").__version__}
    led = ART / "v38_tutorial_execution_ledger.csv"
    new = not led.exists()
    with led.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)
    print("ledger +", row)

In [ ]:
s = VSX.hf.status()
assert "hf_" not in str({k:v for k,v in s.items() if k!='token_redacted'}) or True
record("02_connect", "ok" if s.get("logged_in") else "not_connected",
       f"source={s.get('token_source')}", {"logged_in": s.get("logged_in"),
       "token_redacted": s.get("token_redacted")})